# Title of Notebook

In [12]:
# -----------------------------------------------------------
# Load Melbourne housing dataset into a pandas DataFrame
# -----------------------------------------------------------
# Purpose:
#   - Load the CSV file located at "data/Melbourn_housing.csv"
#   - Store it in a DataFrame named `df`
#   - Perform a basic validation to ensure the file exists
#   - Print minimal confirmation for verification
# -----------------------------------------------------------

from pathlib import Path
import pandas as pd

# Define the path to the CSV file
csv_path = Path("data/Melbourn_housing.csv")

# Validate that the file exists before attempting to load it
if not csv_path.exists():
    raise FileNotFoundError(f"The file was not found at: {csv_path.resolve()}")

# Load the CSV file into a DataFrame
df = pd.read_csv(csv_path)

# Optional: basic verification output
print("Dataset successfully loaded.")
print("Shape of dataframe:", df.shape)
print("First 5 rows:")
print(df.head())

Dataset successfully loaded.
Shape of dataframe: (44413, 21)
First 5 rows:
       Suburb             Address  Rooms Type      Price Method SellerG  \
0  Abbotsford       68 Studley St      2    h        NaN     SS  Jellis   
1  Abbotsford        85 Turner St      2    h  1480000.0      S  Biggin   
2  Abbotsford     25 Bloomburg St      2    h  1035000.0      S  Biggin   
3  Abbotsford  18/659 Victoria St      3    u        NaN     VB  Rounds   
4  Abbotsford        5 Charles St      3    h  1465000.0     SP  Biggin   

        Date  Distance  Postcode  ...  Bathroom  Car  Landsize  BuildingArea  \
0  3/09/2016       2.5    3067.0  ...       1.0  1.0     126.0           NaN   
1  3/12/2016       2.5    3067.0  ...       1.0  1.0     202.0           NaN   
2  4/02/2016       2.5    3067.0  ...       1.0  0.0     156.0          79.0   
3  4/02/2016       2.5    3067.0  ...       2.0  1.0       0.0           NaN   
4  4/03/2017       2.5    3067.0  ...       2.0  0.0     134.0         150

In [13]:
# -----------------------------------------------------------
# Delete the column "Method" from the existing DataFrame `df`
# -----------------------------------------------------------
# Purpose:
#   - Remove the column named "Method" from df
#   - Perform a safety check to avoid errors if the column
#     does not exist
# -----------------------------------------------------------

# Check whether the column exists before attempting to drop it
if "Method" in df.columns:
    df.drop(columns=["Method"], inplace=True)
    print('Column "Method" successfully deleted.')
else:
    print('Column "Method" does not exist in the DataFrame.')

Column "Method" successfully deleted.


In [14]:
# -----------------------------------------------------------
# Remove rows where the column "Price" contains missing values (NA)
# -----------------------------------------------------------
# Purpose:
#   - Ensure that all remaining rows have a valid value in "Price"
#   - Modify the existing DataFrame `df` in place
#   - Provide feedback on how many rows were removed
# -----------------------------------------------------------

# Store the initial number of rows for comparison
initial_row_count = df.shape[0]

# Drop rows where "Price" is NA
df.dropna(subset=["Price"], inplace=True)

# Calculate how many rows were removed
final_row_count = df.shape[0]
rows_removed = initial_row_count - final_row_count

print(f"Rows removed due to missing Price: {rows_removed}")
print(f"Remaining rows: {final_row_count}")

Rows removed due to missing Price: 9749
Remaining rows: 34664


In [15]:
# -----------------------------------------------------------
# Create "Landsize in square feet" from "Landsize" (m² → ft²)
# -----------------------------------------------------------
# Purpose:
#   - Convert the existing "Landsize" column from square meters
#     to square feet and store it in a new column.
# Notes:
#   - 1 square meter = 10.7639104167 square feet
#   - We coerce non-numeric values to NaN to avoid conversion errors
# -----------------------------------------------------------

SQM_TO_SQFT = 10.7639104167  # conversion factor

# Ensure "Landsize" is numeric (invalid parsing becomes NaN)
df["Landsize"] = pd.to_numeric(df["Landsize"], errors="coerce")

# Create the new converted column
df["Landsize in square feet"] = df["Landsize"] * SQM_TO_SQFT

print('Created column: "Landsize in square feet"')
print(df[["Landsize", "Landsize in square feet"]].head(10))

Created column: "Landsize in square feet"
    Landsize  Landsize in square feet
1      202.0              2174.309904
2      156.0              1679.170025
4      134.0              1442.363996
5       94.0              1011.807579
6      120.0              1291.669250
10     181.0              1948.267785
11     245.0              2637.158052
14     256.0              2755.561067
15       NaN                      NaN
16       NaN                      NaN


In [16]:
# -----------------------------------------------------------
# Move "Landsize in square feet" to be immediately right of "Landsize"
# -----------------------------------------------------------
# Purpose:
#   - Reorder columns so the new square-feet column sits directly
#     after the original "Landsize" column for easier comparison.
# -----------------------------------------------------------

new_col = "Landsize in square feet"
base_col = "Landsize"

# Only reorder if both columns exist
if base_col in df.columns and new_col in df.columns:
    cols = list(df.columns)

    # Remove the new column from its current position
    cols.remove(new_col)

    # Insert it right after the base column
    base_index = cols.index(base_col)
    cols.insert(base_index + 1, new_col)

    # Apply the new column order
    df = df[cols]

    print(f'Moved "{new_col}" to the right of "{base_col}".')
else:
    print(f'Cannot reorder: missing "{base_col}" and/or "{new_col}" in df.columns.')

Moved "Landsize in square feet" to the right of "Landsize".


In [17]:
# -----------------------------------------------------------
# Split "Address" into:
#   1) "Address Number" (e.g., "12", "12A", "12/3")
#   2) "Address Name"   (the rest of the address text)
# -----------------------------------------------------------
# Requirements handled:
# - Address number can be:
#   - digits only:            "12"
#   - digits + letter:        "12A" (also supports lowercase: "12a")
#   - digits + "/" + digits:  "12/3"
# - If an address doesn't match the expected patterns, the number will be NA
#   and the full address will remain in "Address Name".
# -----------------------------------------------------------

import re
import numpy as np

# Ensure Address is string (keeps NaN as NaN, and strips whitespace)
address_series = df["Address"].astype("string").str.strip()

# Regex explanation:
# ^\s*                         -> start, optional whitespace
# (?P<num>\d+(?:[A-Za-z]|/\d+)?) -> number:
#                                 - \d+          : one or more digits
#                                 - (?:[A-Za-z]|/\d+)? : optional suffix: letter OR "/digits"
# (?:\s+(?P<name>.*))?         -> optional whitespace + the remaining address
pattern = r"^\s*(?P<num>\d+(?:[A-Za-z]|/\d+)?)\s*(?P<name>.*)$"

extracted = address_series.str.extract(pattern, flags=re.UNICODE)

# Create the two new columns
df["Address Number"] = extracted["num"].where(address_series.notna(), pd.NA)

# If "name" is empty, set it to NA (helps keep data clean)
name_clean = extracted["name"].str.strip()
df["Address Name"] = name_clean.replace("", pd.NA)

# If Address was missing, keep these missing too
df.loc[address_series.isna(), ["Address Number", "Address Name"]] = pd.NA

print('Created columns: "Address Number" and "Address Name"')
print(df[["Address", "Address Number", "Address Name"]].head(15))

Created columns: "Address Number" and "Address Name"
                Address Address Number   Address Name
1          85 Turner St             85      Turner St
2       25 Bloomburg St             25   Bloomburg St
4          5 Charles St              5     Charles St
5      40 Federation La             40  Federation La
6           55a Park St            55a        Park St
10       129 Charles St            129     Charles St
11         124 Yarra St            124       Yarra St
14        98 Charles St             98     Charles St
15     217 Langridge St            217   Langridge St
16      18a Mollison St            18a    Mollison St
17   6/241 Nicholson St          6/241   Nicholson St
18        10 Valiant St             10     Valiant St
19  403/609 Victoria St        403/609    Victoria St
21    25/84 Trenerry Cr          25/84    Trenerry Cr
22    106/119 Turner St        106/119      Turner St


In [18]:
# -----------------------------------------------------------
# Calculate distance from each property to a fixed reference
# point: (-37.79960, 144.99840)
# -----------------------------------------------------------
# Purpose:
#   - Compute great-circle distance using the Haversine formula
#   - Create three new columns:
#       1) Distance (km)
#       2) Distance (miles)
#       3) Distance (light-years)
#
# Notes:
#   - Uses Earth radius = 6,371 km
#   - 1 km = 0.621371 miles
#   - 1 light-year = 9.4607e12 km
# -----------------------------------------------------------

import numpy as np

# Reference coordinates
REF_LAT = -37.79960
REF_LON = 144.99840

# Ensure latitude and longitude columns are numeric
df["Lattitude"] = pd.to_numeric(df["Lattitude"], errors="coerce")
df["Longtitude"] = pd.to_numeric(df["Longtitude"], errors="coerce")

# Convert degrees to radians
lat1 = np.radians(df["Lattitude"])
lon1 = np.radians(df["Longtitude"])
lat2 = np.radians(REF_LAT)
lon2 = np.radians(REF_LON)

# Haversine formula
dlat = lat2 - lat1
dlon = lon2 - lon1

a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
c = 2 * np.arcsin(np.sqrt(a))

EARTH_RADIUS_KM = 6371.0
distance_km = EARTH_RADIUS_KM * c

# Add new columns
df["Distance (km)"] = distance_km
df["Distance (miles)"] = distance_km * 0.621371
df["Distance (light-years)"] = distance_km / 9.4607e12

print("Distance columns created:")
print(df[["Distance (km)", "Distance (miles)", "Distance (light-years)"]].head())

Distance columns created:
   Distance (km)  Distance (miles)  Distance (light-years)
1       0.000000          0.000000            0.000000e+00
2       1.022129          0.635121            1.080395e-13
4       1.134397          0.704881            1.199063e-13
5       0.327881          0.203736            3.465713e-14
6       0.925681          0.575191            9.784485e-14


In [19]:
df

,Suburb,Address,Rooms,Type,Price,SellerG,Date,Distance,Postcode,Bedroom2,...,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Address Number,Address Name,Distance (km),Distance (miles),Distance (light-years)
1,Abbotsford,85 Turner St,2,h,1480000.0,Biggin,3/12/2016,2.5,3067.0,2.0,...,Yarra City Council,-37.79960,144.99840,Northern Metropolitan,4019.0,85,Turner St,0.000000,0.000000,0.000000e+00
2,Abbotsford,25 Bloomburg St,2,h,1035000.0,Biggin,4/02/2016,2.5,3067.0,2.0,...,Yarra City Council,-37.80790,144.99340,Northern Metropolitan,4019.0,25,Bloomburg St,1.022129,0.635121,1.080395e-13
4,Abbotsford,5 Charles St,3,h,1465000.0,Biggin,4/03/2017,2.5,3067.0,3.0,...,Yarra City Council,-37.80930,144.99440,Northern Metropolitan,4019.0,5,Charles St,1.134397,0.704881,1.199063e-13
5,Abbotsford,40 Federation La,3,h,850000.0,Biggin,4/03/2017,2.5,3067.0,3.0,...,Yarra City Council,-37.79690,144.99690,Northern Metropolitan,4019.0,40,Federation La,0.327881,0.203736,3.465713e-14
6,Abbotsford,55a Park St,4,h,1600000.0,Nelson,4/06/2016,2.5,3067.0,3.0,...,Yarra City Council,-37.80720,144.99410,Northern Metropolitan,4019.0,55a,Park St,0.925681,0.575191,9.784485e-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44408,Yarraville,13 Burns St,4,h,1480000.0,Jas,24/02/2018,6.3,3013.0,4.0,...,Maribyrnong City Council,-37.81053,144.88467,Western Metropolitan,6543.0,13,Burns St,10.065416,6.254358,1.063919e-12
44409,Yarraville,29A Murray St,2,h,888000.0,Sweeney,24/02/2018,6.3,3013.0,2.0,...,Maribyrnong City Council,-37.81551,144.88826,Western Metropolitan,6543.0,29A,Murray St,9.836443,6.112080,1.039716e-12
44410,Yarraville,147A Severn St,2,t,705000.0,Jas,24/02/2018,6.3,3013.0,2.0,...,Maribyrnong City Council,-37.82286,144.87856,Western Metropolitan,6543.0,147A,Severn St,10.840740,6.736121,1.145871e-12
44411,Yarraville,12/37 Stephen St,3,h,1140000.0,hockingstuart,24/02/2018,6.3,3013.0,NaN,...,Maribyrnong City Council,NaN,NaN,Western Metropolitan,6543.0,12/37,Stephen St,NaN,NaN,NaN


In [20]:
# -----------------------------------------------------------
# Save the prepared DataFrame to:
#   1) CSV  -> "melbourne-prep.csv"
#   2) Excel -> "melbourn-prep.xlsx"
# -----------------------------------------------------------
# Purpose:
#   - Persist the cleaned and enriched dataset
#   - CSV for portability
#   - Excel for business users
# Notes:
#   - Excel export uses openpyxl as required
# -----------------------------------------------------------

from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ---------- Save as CSV ----------
csv_output_path = "melbourne-prep.csv"
df.to_csv(csv_output_path, index=False)

print(f"CSV file saved: {csv_output_path}")



CSV file saved: melbourne-prep.csv


In [21]:
# -----------------------------------------------------------
# Create an interactive map of all properties (Lat/Lon points)
# with bubble color driven by the "Type" column
# -----------------------------------------------------------
# Output:
#   - An HTML map file saved locally: "melbourne_map_by_type.html"
#
# Notes:
#   - Uses Folium (Leaflet) for an interactive web map
#   - Each row becomes a circle marker ("bubble")
#   - Color is assigned per unique value in df["Type"]
# -----------------------------------------------------------

import numpy as np
import pandas as pd

# Folium is not part of the Python standard library.
# If you haven't installed it yet, run in your VS Code terminal:
#   pip install folium
import folium

# --- Safety: ensure required columns exist ---
required_cols = ["Lattitude", "Longtitude", "Type"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns in df: {missing}")

# --- Clean / validate coordinates ---
df["Lattitude"] = pd.to_numeric(df["Lattitude"], errors="coerce")
df["Longtitude"] = pd.to_numeric(df["Longtitude"], errors="coerce")

# Keep only rows with valid coordinates
df_map = df.dropna(subset=["Lattitude", "Longtitude"]).copy()

if df_map.empty:
    raise ValueError("No valid latitude/longitude rows available to plot.")

# --- Map center: average coordinates (simple, robust default) ---
center_lat = float(df_map["Lattitude"].mean())
center_lon = float(df_map["Longtitude"].mean())

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, control_scale=True)

# --- Assign colors per Type ---
# Folium supports a limited named color set. We'll cycle through a palette.
palette = [
    "red", "blue", "green", "purple", "orange", "darkred", "lightred",
    "beige", "darkblue", "darkgreen", "cadetblue", "darkpurple",
    "white", "pink", "lightblue", "lightgreen", "gray", "black", "lightgray"
]

types = sorted(df_map["Type"].dropna().astype(str).unique())
type_to_color = {t: palette[i % len(palette)] for i, t in enumerate(types)}

# Optional: create a feature group per type so you can toggle visibility
for t in types:
    fg = folium.FeatureGroup(name=f"Type: {t}", show=True)
    sub = df_map[df_map["Type"].astype(str) == t]

    for _, row in sub.iterrows():
        folium.CircleMarker(
            location=[row["Lattitude"], row["Longtitude"]],
            radius=4,                  # bubble size
            color=type_to_color[t],    # stroke color
            fill=True,
            fill_color=type_to_color[t],
            fill_opacity=0.7,
            weight=1,
            # Minimal tooltip for quick inspection (expand as you like)
            tooltip=f"Type: {t}"
        ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# --- Save map to HTML file ---
output_html = "melbourne_map_by_type.html"
m.save(output_html)

print(f"Map saved to: {output_html}")
print("Open this file in your browser to view the interactive map.")

Map saved to: melbourne_map_by_type.html
Open this file in your browser to view the interactive map.
